# Crime PINN — full pipeline, saved to Google Drive

Everything lives in Drive, so a disconnect costs you nothing. After any restart,
run **cell 1 only** and continue where you left off.

**Runtime → Change runtime type → T4 GPU**, then **Run all**. About 60 minutes.

| step | what it does | GPU |
|---|---|---|
| 1 | mount Drive, set the working folder | no |
| 2 | write the four scripts | no |
| 3 | build the burglary field (5 yrs, 24x24, weekly) | no |
| 4 | is there week-to-week signal at all? | no |
| 5 | 15 training runs: physics / physics+MSE / control x 5 seeds | **yes** |
| 6 | final table with mean ± std and t-tests | no |


## 1. Drive — run this first, and after every restart

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
W = '/content/drive/MyDrive/crime_pinn'
os.makedirs(W + '/data', exist_ok=True)
os.makedirs(W + '/results', exist_ok=True)
os.chdir(W)
print('working in', os.getcwd())
print('existing files:', sorted(os.listdir('.')))
import torch; print('GPU:', torch.cuda.is_available())

## 2. Write the scripts

These only save files. Expect `Writing ...` and nothing else.

In [ ]:
%%writefile build_field.py
"""
STEP 1 for the PINN — turn point crime records into a continuous field u(x, y, t)
================================================================================
A PINN solves a PDE, and a PDE acts on a FIELD: a quantity defined over
continuous space and time.  The dataset used so far is a table
(neighbourhood x category x day), which a PDE cannot touch.  This script
rebuilds the raw data into the shape a PDE needs.

What it produces
----------------
    U        (T, H, W)  crime intensity per grid cell per time step
    x, y     (H, W)     cell centre coordinates, normalised to [-1, 1]
    t        (T,)       time, normalised to [0, 1]
    mask     (H, W)     True where the cell is inside the city
    area     (H, W)     Chicago community area id per cell  (77 areas)
    group    (H, W)     'Head' / 'Mid' / 'Tail' per cell, by total crime

Why the last two matter: they carry the fairness analysis over to the PINN.
Without them you can measure accuracy but not who the model serves well.

Design choices worth knowing
----------------------------
* Coordinates are normalised. PINNs train badly on raw latitude/longitude
  because the derivative scales are tiny; [-1, 1] is standard practice.
* Optional Gaussian smoothing. Raw daily counts per cell are extremely spiky,
  and a PDE describes a smooth field. Smoothing is a modelling assumption and
  is recorded in the output so it can be reported honestly.
* Cells that never see a crime all year are outside the city boundary and are
  masked out, not treated as true zeros.
* The train/val/test split is by TIME, never random, matching the rest of the
  project. Random splits would leak the future into training.

Usage
-----
    python build_field.py --year 2015 --grid 48 --freq D --smooth 1.0
    python build_field.py --year 2015 --grid 64 --freq W --smooth 0.8
"""
from __future__ import annotations

import argparse
import json
import time
from urllib.parse import urlencode

import numpy as np
import pandas as pd

RESOURCE = "https://data.cityofchicago.org/resource/ijzp-q8t2.json"
# Chicago bounding box, trimmed to the mainland city (a few records land in
# the lake or are mis-geocoded to 0,0 -- those are dropped).
LAT_LO, LAT_HI = 41.644, 42.023
LON_LO, LON_HI = -87.940, -87.524


# --------------------------------------------------------------------------- #
# 1. fetch
# --------------------------------------------------------------------------- #
def fetch(year0: int, year1: int, category: str | None = None,
          page: int = 50_000, max_rows: int = 2_000_000) -> pd.DataFrame:
    """Page the Socrata endpoint for a RANGE of years.

    The category filter is pushed server-side. Without it, five years of all
    crime types is ~1.3M rows; with it, burglary alone is ~65k. That is the
    difference between a 30-second fetch and a 20-minute one.
    """
    where = (f"date >= '{year0}-01-01T00:00:00.000' "
             f"AND date <= '{year1}-12-31T23:59:59.000' "
             f"AND latitude IS NOT NULL")
    if category:
        where += f" AND upper(primary_type) = '{category.upper()}'"
    out, offset = [], 0
    while offset < max_rows:
        q = urlencode({"$select": "date,latitude,longitude,community_area,primary_type",
                       "$where": where, "$limit": page, "$offset": offset})
        chunk = pd.read_json(f"{RESOURCE}?{q}")
        if chunk.empty:
            break
        out.append(chunk)
        offset += page
        print(f"  fetched {offset} rows...", flush=True)
        time.sleep(0.4)
    if not out:
        raise SystemExit("No rows returned - check the year or the portal.")
    return pd.concat(out, ignore_index=True)


# --------------------------------------------------------------------------- #
# 2. smoothing without scipy (separable Gaussian, reflect padding)
# --------------------------------------------------------------------------- #
def gaussian_blur(A: np.ndarray, sigma: float) -> np.ndarray:
    """Blur the last two axes of A. sigma is in grid cells; 0 disables."""
    if sigma <= 0:
        return A
    r = max(1, int(round(3 * sigma)))
    k = np.exp(-0.5 * (np.arange(-r, r + 1) / sigma) ** 2)
    k /= k.sum()
    out = A.astype(np.float64)
    for axis in (-2, -1):
        out = np.apply_along_axis(
            lambda v: np.convolve(np.pad(v, r, mode="reflect"), k, mode="valid"),
            axis, out)
    return out


# --------------------------------------------------------------------------- #
# 3. build
# --------------------------------------------------------------------------- #
def build(args):
    df = (pd.read_csv(args.csv) if args.csv
          else fetch(args.year0, args.year1, args.category))
    n0 = len(df)
    df = df.dropna(subset=["date", "latitude", "longitude"])
    df["latitude"] = pd.to_numeric(df["latitude"], errors="coerce")
    df["longitude"] = pd.to_numeric(df["longitude"], errors="coerce")
    df = df.dropna(subset=["latitude", "longitude"])
    df = df[(df.latitude.between(LAT_LO, LAT_HI)) &
            (df.longitude.between(LON_LO, LON_HI))]
    df["date"] = pd.to_datetime(df["date"], errors="coerce")
    df = df.dropna(subset=["date"])
    if args.category:                       # optional single-category field
        df = df[df["primary_type"].str.upper() == args.category.upper()]
    print(f"\nkept {len(df)} of {n0} records "
          f"({100*len(df)/max(n0,1):.1f}%) after cleaning")

    # ---- spatial grid ----------------------------------------------------- #
    G = args.grid
    lat_edges = np.linspace(LAT_LO, LAT_HI, G + 1)
    lon_edges = np.linspace(LON_LO, LON_HI, G + 1)
    iy = np.clip(np.digitize(df.latitude.values, lat_edges) - 1, 0, G - 1)
    ix = np.clip(np.digitize(df.longitude.values, lon_edges) - 1, 0, G - 1)

    # ---- temporal bins ---------------------------------------------------- #
    per = df["date"].dt.to_period({"D": "D", "W": "W", "M": "M"}[args.freq])
    periods = np.array(sorted(per.unique()))
    tindex = {p: i for i, p in enumerate(periods)}
    it = per.map(tindex).values
    T = len(periods)
    fname = {"D": "daily", "W": "weekly", "M": "monthly"}[args.freq]
    print(f"grid {G}x{G} cells | {T} time steps ({fname})")
    if int(T * .70) < 100:
        print(f"  WARNING: only {int(T*.70)} training time steps. A PDE needs a"
              f"\n  time derivative -- aim for 100+. Widen --year0/--year1 or use --freq D.")

    # ---- counts ----------------------------------------------------------- #
    U = np.zeros((T, G, G), dtype=np.float64)
    np.add.at(U, (it, iy, ix), 1.0)

    # ---- mask: cells that never see a crime are outside the city ---------- #
    total = U.sum(0)
    mask = total >= args.min_events
    print(f"in-city cells: {mask.sum()} of {G*G} "
          f"({100*mask.sum()/(G*G):.1f}%)  [min_events={args.min_events}]")

    # ---- community area per cell (modal) ---------------------------------- #
    area = np.full((G, G), -1, dtype=np.int32)
    ca = pd.to_numeric(df["community_area"], errors="coerce")
    ok = ca.notna().values
    tmp = pd.DataFrame({"iy": iy[ok], "ix": ix[ok], "ca": ca[ok].astype(int).values})
    modal = tmp.groupby(["iy", "ix"])["ca"].agg(lambda s: s.value_counts().idxmax())
    for (yy, xx), v in modal.items():
        area[yy, xx] = v

    # ---- Head / Mid / Tail per cell, by total crime (20/30/50) ------------ #
    group = np.full((G, G), "", dtype=object)
    idx = np.argwhere(mask)
    vals = total[mask]
    order = np.argsort(vals)[::-1]
    n = len(order); nh = max(1, round(n * .20)); nm = max(1, round(n * .30))
    for rank, o in enumerate(order):
        yy, xx = idx[o]
        group[yy, xx] = "Head" if rank < nh else ("Mid" if rank < nh + nm else "Tail")

    # ---- smoothing -------------------------------------------------------- #
    U_raw = U.copy()
    U = gaussian_blur(U, args.smooth)
    U *= mask[None, :, :]

    # ---- normalised coordinates ------------------------------------------- #
    latc = 0.5 * (lat_edges[:-1] + lat_edges[1:])
    lonc = 0.5 * (lon_edges[:-1] + lon_edges[1:])
    yy, xx = np.meshgrid(latc, lonc, indexing="ij")
    xn = 2 * (xx - LON_LO) / (LON_HI - LON_LO) - 1
    yn = 2 * (yy - LAT_LO) / (LAT_HI - LAT_LO) - 1
    tn = np.linspace(0.0, 1.0, T)

    # ---- time split (never random) ---------------------------------------- #
    ntr, nva = int(T * .70), int(T * .10)
    split = np.array(["train"] * ntr + ["val"] * nva + ["test"] * (T - ntr - nva))

    # ---- report ----------------------------------------------------------- #
    inside = U[:, mask]
    print(f"\nfield summary")
    print(f"  U shape           {U.shape}")
    print(f"  mean intensity    {inside.mean():.4f} events/cell/step")
    print(f"  max  intensity    {inside.max():.2f}")
    print(f"  empty cell-steps  {100*(U_raw[:, mask] == 0).mean():.1f}% "
          f"(before smoothing)")
    for g in ("Head", "Mid", "Tail"):
        m = (group == g)
        print(f"  {g:5s} {m.sum():4d} cells   mean {U[:, m].mean():.4f}")
    print(f"  split             train {ntr} / val {nva} / test {T-ntr-nva} steps")

    np.savez_compressed(
        args.out, U=U.astype(np.float32), U_raw=U_raw.astype(np.float32),
        x=xn.astype(np.float32), y=yn.astype(np.float32), t=tn.astype(np.float32),
        mask=mask, area=area, group=group.astype("U4"), split=split,
        lat_edges=lat_edges, lon_edges=lon_edges,
        meta=json.dumps({"years": [args.year0, args.year1], "grid": G, "freq": args.freq,
                         "smooth": args.smooth, "min_events": args.min_events,
                         "category": args.category, "n_records": int(len(df)),
                         "domain": {"x": [-1, 1], "y": [-1, 1], "t": [0, 1]}}))
    print(f"\nsaved -> {args.out}")
    print("domain for PDE collocation points: x,y in [-1,1], t in [0,1]")


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--year0", type=int, default=2015, help="first year (inclusive)")
    ap.add_argument("--year1", type=int, default=None, help="last year; default = year0")
    ap.add_argument("--grid", type=int, default=48, help="cells per side")
    ap.add_argument("--freq", default="D", choices=["D", "W", "M"])
    ap.add_argument("--smooth", type=float, default=1.0,
                    help="Gaussian sigma in cells; 0 = none")
    ap.add_argument("--min-events", type=int, default=5,
                    help="a cell needs this many events all year to count as in-city")
    ap.add_argument("--category", default=None,
                    help="e.g. BURGLARY - the Short et al. model is a burglary model")
    ap.add_argument("--csv", default=None, help="use a local CSV instead of the API")
    ap.add_argument("--out", default="data/chi_field.npz")
    args = ap.parse_args()
    if args.year1 is None:
        args.year1 = args.year0
    import os
    os.makedirs(os.path.dirname(args.out) or ".", exist_ok=True)
    build(args)


if __name__ == "__main__":
    import sys
    if "ipykernel" in sys.modules:
        print("=" * 70)
        print("This file was RUN as a notebook cell instead of being SAVED.")
        print("Add this as the FIRST line of this cell, then press Enter:")
        print()
        print("    %%writefile build_field.py")
        print()
        print("Then run it with:  !python build_field.py --year 2015 ...")
        print("=" * 70)
    else:
        main()


In [ ]:
%%writefile crime_pinn.py
"""
STEP 2 — a Physics-Informed Neural Network for the Short et al. crime model
================================================================================
THE PHYSICS
-----------
Short, D'Orsogna, Pasour, Tita, Brantingham, Bertozzi & Chayes (2008),
"A statistical model of criminal behavior", Math. Models Methods Appl. Sci. 18.

Two coupled fields on the city:

    A(x,y,t)   attractiveness  - how appealing a location is to burgle
    rho(x,y,t) offender density

    dA/dt   = eta * lap(A) - A + A0 + rho*A
    drho/dt = div( grad(rho) - 2*(rho/A)*grad(A) ) - rho*A + A - A0

Reading the terms:
  eta*lap(A)          attractiveness spreads to neighbouring locations
  -A + A0             it decays back to a baseline A0
  +rho*A              a crime raises local attractiveness (repeat victimisation)
  -2*(rho/A)*grad(A)  offenders drift UP the attractiveness gradient
  -rho*A              an offender leaves after committing a crime
  +A - A0             replacement of offenders

The OBSERVED quantity is the crime rate, which in this model is  rho * A.
That is what our field U from build_field.py measures.

Self-consistency: at a uniform steady state both equations give rho*A = A - A0,
so the system is consistent (a useful check that we transcribed it correctly).

WHAT THIS SCRIPT DOES
---------------------
* network  (x, y, t) -> (A, rho), both forced positive by softplus
* data loss     :  (rho*A - U_observed)^2  on training time steps
* physics loss  :  the two PDE residuals at random collocation points
* eta and A0 are LEARNED, not fixed - the model discovers them from Chicago
* --pde-weight 0 gives the capacity-matched control: identical network, no physics
* evaluation reports F1 AND AUC per Head/Mid/Tail group, because this project
  has already shown that F1 alone can move without any gain in real skill

WHAT IT DOES NOT DO
-------------------
It does not claim the physics is a correct description of crime. The
attractiveness mechanism comes from Broken Windows theory, which is contested
(Sampson & Raudenbush 2004; Goodson & Hoyer-Leitzel 2021). This code lets you
MEASURE what that assumption does to predictions, particularly in low-crime
neighbourhoods. That measurement is the point.

Usage
-----
    python crime_pinn.py --data data/burg_w24.npz --pde-weight 1.0 --epochs 8000
    python crime_pinn.py --data data/burg_w24.npz --pde-weight 0.0 --epochs 8000
"""
from __future__ import annotations

import argparse
import json
import os
import random
import time

import numpy as np
import torch
import torch.nn as nn
from sklearn.metrics import roc_auc_score


# --------------------------------------------------------------------------- #
# determinism  (this project learned the hard way that seeding torch is not enough)
# --------------------------------------------------------------------------- #
def seed_everything(seed: int):
    os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    try:
        torch.use_deterministic_algorithms(True, warn_only=True)
    except TypeError:
        torch.use_deterministic_algorithms(True)


# --------------------------------------------------------------------------- #
# network
# --------------------------------------------------------------------------- #
class FourierFeatures(nn.Module):
    """Random Fourier features. Plain MLPs are biased toward low frequencies and
    struggle to fit sharp hotspots; this is the standard fix (Tancik et al. 2020,
    used for PINNs by Wang et al.)."""
    def __init__(self, in_dim=3, n=32, scale=3.0):
        super().__init__()
        self.register_buffer("B", torch.randn(in_dim, n) * scale)

    def forward(self, z):
        p = 2 * np.pi * z @ self.B
        return torch.cat([torch.sin(p), torch.cos(p)], -1)


class A0Field(nn.Module):
    """A0(x, y): the static baseline attractiveness of a location.

    Short et al. state explicitly that A0 "is not necessarily uniform over the
    lattice grids". Learning it as a field rather than one constant is therefore
    FAITHFUL to the original model, not an extension. It also gives the network
    a proper place to store the spatial crime map, so the dynamics are free to
    model change over time instead of re-learning geography.
    """
    def __init__(self, width=64, depth=3, fourier=16, f_scale=3.0):
        super().__init__()
        self.ff = FourierFeatures(2, fourier, f_scale) if fourier else None
        d = 2 * fourier if fourier else 2
        L = []
        for _ in range(depth):
            L += [nn.Linear(d, width), nn.Tanh()]; d = width
        L += [nn.Linear(d, 1)]
        self.net = nn.Sequential(*L)

    def forward(self, x, y):
        z = torch.stack([x, y], -1)
        h = self.ff(z) if self.ff is not None else z
        return torch.nn.functional.softplus(self.net(h)[..., 0]) + 1e-3


class CrimePINN(nn.Module):
    """(x, y, t) -> (A, rho).  Both outputs are positive: A appears in a
    denominator (rho/A) and rho is a density."""
    def __init__(self, width=128, depth=5, fourier=32, f_scale=3.0,
                 spatial_A0=True):
        super().__init__()
        self.spatial_A0 = spatial_A0
        self.A0_net = A0Field(64, 3, 16, f_scale) if spatial_A0 else None
        self.ff = FourierFeatures(3, fourier, f_scale) if fourier else None
        d_in = 2 * fourier if fourier else 3
        layers, d = [], d_in
        for _ in range(depth):
            layers += [nn.Linear(d, width), nn.Tanh()]
            d = width
        layers += [nn.Linear(d, 2)]
        self.net = nn.Sequential(*layers)
        # learned physical constants, kept positive through softplus
        self.raw_eta = nn.Parameter(torch.tensor(-1.0))
        self.raw_A0 = nn.Parameter(torch.tensor(0.0))

    @property
    def eta(self): return torch.nn.functional.softplus(self.raw_eta)

    @property
    def A0(self): return torch.nn.functional.softplus(self.raw_A0)

    def A0_at(self, x, y):
        """Baseline attractiveness: a field if enabled, otherwise the scalar."""
        if self.A0_net is not None:
            return self.A0_net(x, y)
        return self.A0.expand_as(x)

    def forward(self, x, y, t):
        z = torch.stack([x, y, t], -1)
        h = self.ff(z) if self.ff is not None else z
        out = self.net(h)
        A = torch.nn.functional.softplus(out[..., 0]) + 1e-3   # strictly > 0
        rho = torch.nn.functional.softplus(out[..., 1])
        return A, rho


# --------------------------------------------------------------------------- #
# PDE residuals
# --------------------------------------------------------------------------- #
def grad(f, v):
    return torch.autograd.grad(f, v, torch.ones_like(f), create_graph=True)[0]


def smooth_residuals(model, x, y, t):
    """ABLATION: a generic smoothness prior with NO burglary content.

    Penalises how fast the predicted crime rate changes in time and how curved
    it is in space. If this reproduces the gain from the Short PDE, then the
    physics is doing nothing a plain regulariser could not do -- which is the
    control this project has learned to always build.
    """
    x = x.requires_grad_(True); y = y.requires_grad_(True); t = t.requires_grad_(True)
    A, rho = model(x, y, t)
    u = rho * A
    u_t = grad(u, t)
    u_x, u_y = grad(u, x), grad(u, y)
    lap = grad(u_x, x) + grad(u_y, y)
    return u_t, lap


def pde_residuals(model, x, y, t):
    """Both residuals of the Short et al. system. Zero when the physics holds."""
    x = x.requires_grad_(True); y = y.requires_grad_(True); t = t.requires_grad_(True)
    A, rho = model(x, y, t)

    A_t = grad(A, t)
    A_x, A_y = grad(A, x), grad(A, y)
    A_xx, A_yy = grad(A_x, x), grad(A_y, y)
    lapA = A_xx + A_yy

    rho_t = grad(rho, t)
    rho_x, rho_y = grad(rho, x), grad(rho, y)
    rho_xx, rho_yy = grad(rho_x, x), grad(rho_y, y)
    lap_rho = rho_xx + rho_yy

    # div( (rho/A) grad A ) = grad(rho/A) . grad A + (rho/A) lap A
    w = rho / A
    w_x, w_y = grad(w, x), grad(w, y)
    div_term = w_x * A_x + w_y * A_y + w * lapA

    eta = model.eta
    A0 = model.A0_at(x, y)          # spatial baseline, per Short et al.
    rA = rho * A
    rA_res = A_t - (eta * lapA - A + A0 + rA)
    rr_res = rho_t - (lap_rho - 2.0 * div_term - rA + A - A0)
    return rA_res, rr_res


# --------------------------------------------------------------------------- #
# data
# --------------------------------------------------------------------------- #
def load(path, device):
    d = np.load(path, allow_pickle=True)
    U, mask, group = d["U"], d["mask"], d["group"]
    # U is Gaussian-smoothed so the PDE has a differentiable field to act on.
    # For BINARY labels we must use the RAW counts: "did a burglary actually
    # happen in this cell this week". Using the smoothed field makes almost
    # every cell non-zero and the labels degenerate.
    U_raw = d["U_raw"]
    x2, y2, t1, split = d["x"], d["y"], d["t"], d["split"]
    T = U.shape[0]

    # scale so the field is O(1): the PDE is written in nondimensional units
    scale = float(U[:, mask].mean())
    Us = U / max(scale, 1e-8)
    # for the Poisson likelihood we need the RATE on the raw count scale, so we
    # keep the conversion factor: rate = raw_scale * (rho*A)
    raw_scale = float(U_raw[:, mask].mean())

    iy, ix = np.where(mask)
    XY = np.stack([x2[iy, ix], y2[iy, ix]], 1).astype(np.float32)   # (M,2)
    G = group[iy, ix]

    def pack(which):
        ts = np.where(split == which)[0]
        n = len(ts) * len(iy)
        xs = np.tile(XY[:, 0], len(ts)); ys = np.tile(XY[:, 1], len(ts))
        tt = np.repeat(t1[ts], len(iy))
        rr = np.repeat(ts, len(iy)); cc = np.tile(iy, len(ts)); dd = np.tile(ix, len(ts))
        uu = Us[rr, cc, dd]
        ur = U_raw[rr, cc, dd]
        gg = np.tile(G, len(ts))
        to = lambda a, dt=torch.float32: torch.tensor(a, dtype=dt, device=device)
        return dict(x=to(xs), y=to(ys), t=to(tt), u=to(uu),
                    u_raw=ur.astype(np.float32), u_raw_t=to(ur), g=gg, n=n)

    meta = json.loads(str(d["meta"]))
    # in-city cell centres, so collocation points never land in the lake
    city = torch.tensor(XY, dtype=torch.float32, device=device)
    return (pack("train"), pack("val"), pack("test"), scale, meta,
            raw_scale, city)


# --------------------------------------------------------------------------- #
# evaluation — F1 AND AUC per group
# --------------------------------------------------------------------------- #
def _predict(model, S, raw_scale=1.0):
    """Predicted burglary RATE per cell per week, on the raw count scale."""
    model.eval()
    with torch.no_grad():
        A, rho = model(S["x"], S["y"], S["t"])
        return (raw_scale * rho * A).cpu().numpy()


def tune_threshold(model, S, raw_scale=1.0):
    """Pick the threshold that maximises F1 on VALIDATION.

    Each model gets its own threshold, which is what a deployer would do. The
    positive-prediction rate is reported alongside every score so that a change
    driven by the operating point cannot be mistaken for a change in skill --
    the failure mode documented earlier in this project.
    """
    p = _predict(model, S, raw_scale)
    y = (S["u_raw"] > 0).astype(int)
    if y.sum() in (0, len(y)):
        return 0.5
    best, bt = -1.0, 0.5
    for q in np.linspace(1, 99, 99):
        t = float(np.percentile(p, q))
        pb = (p > t).astype(int)
        tp = ((pb == 1) & (y == 1)).sum(); fp = ((pb == 1) & (y == 0)).sum()
        fn = ((pb == 0) & (y == 1)).sum()
        f1 = 0 if 2*tp+fp+fn == 0 else 2*tp/(2*tp+fp+fn)
        if f1 > best:
            best, bt = f1, t
    return bt


def evaluate(model, S, thr=None, raw_scale=1.0):
    pred = _predict(model, S, raw_scale)
    true = S["u_raw"]                      # compare rate against actual counts
    ybin = (S["u_raw"] > 0).astype(int)          # RAW counts, not smoothed
    res = {"rmse": float(np.sqrt(np.mean((pred - true) ** 2))),
           "mae": float(np.mean(np.abs(pred - true))),
           "base_rate": float(100 * ybin.mean()),
           "thr": float(thr) if thr is not None else float("nan")}
    t = 0.5 if thr is None else thr
    for g in ("Head", "Mid", "Tail", "ALL"):
        m = np.ones_like(ybin, bool) if g == "ALL" else (S["g"] == g)
        yy, pp = ybin[m], np.nan_to_num(pred[m])
        res[f"{g}_base"] = float(100 * yy.mean()) if len(yy) else float("nan")
        res[f"{g}_rmse"] = float(np.sqrt(np.mean((pp - true[m]) ** 2))) if len(yy) else float("nan")
        if len(yy) == 0 or yy.sum() in (0, len(yy)):
            res[f"{g}_f1"] = res[f"{g}_auc"] = res[f"{g}_rate"] = float("nan")
            continue
        pb = (pp > t).astype(int)
        tp = ((pb == 1) & (yy == 1)).sum(); fp = ((pb == 1) & (yy == 0)).sum()
        fn = ((pb == 0) & (yy == 1)).sum()
        res[f"{g}_f1"] = float(0 if 2*tp+fp+fn == 0 else 200*tp/(2*tp+fp+fn))
        res[f"{g}_auc"] = float(100 * roc_auc_score(yy, pp))
        res[f"{g}_rate"] = float(100 * pb.mean())
    res["gap_f1"] = res["Head_f1"] - res["Tail_f1"]
    res["gap_auc"] = res["Head_auc"] - res["Tail_auc"]
    return res


# --------------------------------------------------------------------------- #
# training
# --------------------------------------------------------------------------- #
def train(args):
    seed_everything(args.seed)
    dev = "cuda" if torch.cuda.is_available() else "cpu"
    tr, va, te, scale, meta, raw_scale, city = load(args.data, dev)
    print(f"device {dev} | field scale {scale:.4f} | raw scale {raw_scale:.4f} | "
          f"train pts {tr['n']:,} | test pts {te['n']:,}")
    print(f"data: {meta}")

    model = CrimePINN(args.width, args.depth, args.fourier, args.f_scale,
                      spatial_A0=not args.scalar_A0).to(dev)
    npar = sum(p.numel() for p in model.parameters())
    print(f"model: {npar:,} parameters | physics={args.physics} "
          f"| loss={args.loss} | A0={'field' if not args.scalar_A0 else 'scalar'} "
          f"| pde-weight {args.pde_weight}")

    opt = torch.optim.Adam(model.parameters(), lr=args.lr)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, args.epochs)
    mse = nn.MSELoss()

    # collocation points must lie INSIDE the city. Sampling uniformly over the
    # bounding box would enforce burglary physics over Lake Michigan.
    ncell = city.shape[0]
    jitter = 2.0 / max(meta.get("grid", 24), 1)      # half a cell, in [-1,1] units
    def collocation(n):
        i = torch.randint(0, ncell, (n,), device=dev)
        cx = city[i, 0] + (torch.rand(n, device=dev) - .5) * jitter
        cy = city[i, 1] + (torch.rand(n, device=dev) - .5) * jitter
        return cx.clamp(-1, 1), cy.clamp(-1, 1), torch.rand(n, device=dev)

    # adaptive sampling: keep a large pool, train on the points with the biggest
    # residuals. This is the idea from the Lin & Chen paper (Causal AS).
    pool = collocation(args.pool) if args.sampling == "adaptive" else None

    best, best_state, t0 = float("inf"), None, time.time()
    for ep in range(1, args.epochs + 1):
        model.train(); opt.zero_grad()

        idx = torch.randint(0, tr["n"], (args.batch,), device=dev)
        A, rho = model(tr["x"][idx], tr["y"][idx], tr["t"][idx])
        if args.loss == "poisson":
            # burglaries per cell per week are COUNTS. MSE treats them as
            # Gaussian, which over-weights busy cells -- a Head/Tail bias baked
            # straight into the objective. Poisson is the correct likelihood.
            lam = raw_scale * rho * A + 1e-6
            target = tr["u_raw_t"][idx]
            loss_data = (lam - target * torch.log(lam)).mean()
        else:
            loss_data = mse(rho * A, tr["u"][idx])

        loss_pde = torch.tensor(0.0, device=dev)
        if args.pde_weight > 0 and args.physics != "none":
            if args.sampling == "adaptive" and ep % args.resample == 0:
                fn = pde_residuals if args.physics == "short" else smooth_residuals
                with torch.enable_grad():
                    r1, r2 = fn(model, *[p.clone() for p in pool])
                    sev = (r1.abs() + r2.abs()).detach()
                keep = torch.topk(sev, args.n_coll).indices
                cx, cy, ct = pool[0][keep], pool[1][keep], pool[2][keep]
            else:
                cx, cy, ct = collocation(args.n_coll)
            if args.physics == "short":
                r1, r2 = pde_residuals(model, cx, cy, ct)
            else:                                   # generic smoothness ablation
                r1, r2 = smooth_residuals(model, cx, cy, ct)
            loss_pde = (r1 ** 2).mean() + (r2 ** 2).mean()

        loss = loss_data + args.pde_weight * loss_pde
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        opt.step(); sched.step()

        if ep % args.eval_every == 0 or ep == args.epochs:
            v = evaluate(model, va, raw_scale=raw_scale)
            if v["rmse"] < best:
                best = v["rmse"]
                best_state = {k: t.detach().clone() for k, t in model.state_dict().items()}
            print(f"  ep {ep:6d} | data {loss_data.item():.4f} "
                  f"| pde {loss_pde.item():.4f} | val rmse {v['rmse']:.4f} "
                  f"| eta {model.eta.item():.4f} A0 {model.A0.item():.4f} "
                  f"| {time.time()-t0:.0f}s", flush=True)

    if best_state is not None:
        model.load_state_dict(best_state)
    thr = tune_threshold(model, va, raw_scale)   # tuned on validation
    r = evaluate(model, te, thr, raw_scale)
    r.update(pde_weight=args.pde_weight, seed=args.seed, scale=scale,
             physics=args.physics, loss=args.loss,
             spatial_A0=not args.scalar_A0,
             eta=float(model.eta.item()), A0=float(model.A0.item()),
             params=npar, sampling=args.sampling, data=args.data)

    print("\n" + "=" * 70)
    tag = ("CONTROL (no physics)" if args.physics == "none" or args.pde_weight == 0
           else f"PINN — physics={args.physics}")
    print(f"TEST — {tag}  [loss={args.loss}]")
    print("=" * 70)
    print(f"{'group':>6} {'F1':>8} {'AUC':>8} {'RMSE':>8} {'says yes':>9} {'actual':>8}")
    for g in ("Head", "Mid", "Tail", "ALL"):
        print(f"{g:>6} {r[g+'_f1']:8.2f} {r[g+'_auc']:8.2f} "
              f"{r[g+'_rmse']:8.3f} {r[g+'_rate']:8.1f}% {r[g+'_base']:7.1f}%")
    print(f"\n  threshold {r['thr']:.4f} (tuned on validation)")
    print(f"\n  Head-Tail gap:  F1 {r['gap_f1']:+.2f}   AUC {r['gap_auc']:+.2f}")
    print(f"  learned eta {r['eta']:.4f}   A0 {r['A0']:.4f}")
    print("  eta is the attractiveness diffusion rate the model inferred from data.")
    print("  Compare F1 and AUC gaps: if only F1 moves, the change is threshold,")
    print("  not skill -- the failure mode this project already documented.")

    if args.save:
        os.makedirs(os.path.dirname(args.save) or ".", exist_ok=True)
        with open(args.save, "a") as fh:
            fh.write(json.dumps(r) + "\n")
        print(f"\nappended -> {args.save}")
    return r


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--data", default="data/burg_w24.npz")
    ap.add_argument("--physics", default="short", choices=["short", "smooth", "none"],
                    help="short = Short et al. PDE; smooth = generic smoothness "
                         "ablation with no burglary content; none = control")
    ap.add_argument("--loss", default="poisson", choices=["poisson", "mse"],
                    help="poisson is the correct likelihood for counts")
    ap.add_argument("--scalar-A0", action="store_true",
                    help="revert A0 to a single constant (old behaviour)")
    ap.add_argument("--pde-weight", type=float, default=1.0,
                    help="0 = capacity-matched control with no physics")
    ap.add_argument("--epochs", type=int, default=8000)
    ap.add_argument("--batch", type=int, default=4096)
    ap.add_argument("--n-coll", type=int, default=4096)
    ap.add_argument("--pool", type=int, default=100_000,
                    help="candidate pool for adaptive sampling")
    ap.add_argument("--sampling", default="uniform", choices=["uniform", "adaptive"])
    ap.add_argument("--resample", type=int, default=100)
    ap.add_argument("--width", type=int, default=128)
    ap.add_argument("--depth", type=int, default=5)
    ap.add_argument("--fourier", type=int, default=32, help="0 disables")
    ap.add_argument("--f-scale", type=float, default=3.0)
    ap.add_argument("--lr", type=float, default=1e-3)
    ap.add_argument("--eval-every", type=int, default=500)
    ap.add_argument("--seed", type=int, default=0)
    ap.add_argument("--save", default="results/pinn.jsonl")
    train(ap.parse_args())


if __name__ == "__main__":
    import sys
    if "ipykernel" in sys.modules:
        print("=" * 70)
        print("This file was RUN as a notebook cell instead of being SAVED.")
        print("Add  %%writefile crime_pinn.py  as the FIRST line, then Enter.")
        print("=" * 70)
    else:
        main()


In [ ]:
%%writefile signal_check.py
"""
Is there any week-to-week signal to predict?  (run this BEFORE improving anything)
================================================================================
The PINN's per-group AUC sat near 50, which says it tells neighbourhoods apart
but not weeks apart.  Two explanations:

    (a) the model is not good enough yet        -> worth improving
    (b) the data has no week-to-week signal     -> no model will find it

This script decides between them WITHOUT training anything, by asking how much
of the variance three trivial predictors explain on the test period:

    1. global mean            - one number for everything
    2. per-cell mean          - the spatial map, no dynamics at all
    3. per-cell mean + lag-1  - the map plus last week's deviation

If (3) is barely better than (2), there is nothing temporal to learn and the
ceiling is the data, not the architecture.

    python signal_check.py --data data/burg_w24.npz
"""
import argparse
import numpy as np


def r2(true, pred):
    ss = ((true - true.mean()) ** 2).sum()
    return float(1 - ((true - pred) ** 2).sum() / ss) if ss > 0 else float("nan")


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--data", default="data/burg_w24.npz")
    ap.add_argument("--raw", action="store_true",
                    help="use unsmoothed counts (harder, more honest)")
    a = ap.parse_args()

    d = np.load(a.data, allow_pickle=True)
    U = d["U_raw"] if a.raw else d["U"]
    mask, group, split = d["mask"], d["group"], d["split"]
    iy, ix = np.where(mask)
    X = U[:, iy, ix]                       # (T, cells)
    tr = np.where(split == "train")[0]
    te = np.where(split == "test")[0]

    cell_mean = X[tr].mean(0)              # the spatial map, fit on train only
    glob = X[tr].mean()

    print(f"{'field':>10}: {'raw counts' if a.raw else 'smoothed'}   "
          f"cells {X.shape[1]}   train {len(tr)}w   test {len(te)}w\n")

    # ---- how well can we do with no dynamics at all? ---------------------- #
    yt = X[te]
    p_glob = np.full_like(yt, glob)
    p_map = np.tile(cell_mean, (len(te), 1))
    # lag-1: map + a fraction of last week's deviation from the map
    prev = X[te - 1]
    dev = prev - cell_mean
    # fit the AR coefficient on TRAIN only
    dtr = X[tr[1:]] - cell_mean
    dtr_prev = X[tr[:-1]] - cell_mean
    beta = float((dtr_prev * dtr).sum() / max((dtr_prev ** 2).sum(), 1e-9))
    p_ar = cell_mean + beta * dev

    print(f"{'predictor':>34} {'R2 on test':>11}")
    print("-" * 47)
    print(f"{'1. global mean (one number)':>34} {r2(yt, p_glob):11.4f}")
    print(f"{'2. per-cell mean (the map)':>34} {r2(yt, p_map):11.4f}")
    print(f"{'3. map + last week (AR-1)':>34} {r2(yt, p_ar):11.4f}")
    print(f"\n  fitted lag-1 coefficient beta = {beta:+.4f}")
    gain = r2(yt, p_ar) - r2(yt, p_map)
    print(f"  temporal gain over the plain map = {gain:+.4f} R2")

    # ---- same question, per group ---------------------------------------- #
    G = group[iy, ix]
    print(f"\n{'group':>6} {'cells':>6} {'R2 map':>9} {'R2 +lag1':>9} {'gain':>8}")
    print("-" * 42)
    for g in ("Head", "Mid", "Tail"):
        m = (G == g)
        if m.sum() == 0:
            continue
        a_ = r2(yt[:, m], p_map[:, m]); b_ = r2(yt[:, m], p_ar[:, m])
        print(f"{g:>6} {m.sum():6d} {a_:9.4f} {b_:9.4f} {b_-a_:+8.4f}")

    # ---- lag-1 autocorrelation of the deviation --------------------------- #
    dall = X - cell_mean
    ac = float((dall[:-1] * dall[1:]).sum() /
               max(np.sqrt((dall[:-1] ** 2).sum() * (dall[1:] ** 2).sum()), 1e-9))
    print(f"\n  lag-1 autocorrelation of the deviation-from-map: {ac:+.4f}")

    print("\n" + "=" * 62)
    if gain < 0.01:
        print("VERDICT: essentially NO week-to-week signal beyond the spatial map.")
        print("  Improving the PINN will not create temporal skill that is not")
        print("  in the data. Report the map-learning honestly and stop tuning.")
    elif gain < 0.05:
        print("VERDICT: WEAK temporal signal. Some room, but small. Expect modest")
        print("  gains at best -- set expectations before spending GPU time.")
    else:
        print("VERDICT: real temporal signal exists. The PINN is leaving skill on")
        print("  the table and the Tier-1 changes are worth making.")
    print("=" * 62)


if __name__ == "__main__":
    import sys
    if "ipykernel" in sys.modules:
        print("Add  %%writefile signal_check.py  as the FIRST line of this cell.")
    else:
        main()


In [ ]:
%%writefile analyse_pinn.py
"""
Final PINN table: mean +/- std over seeds, with Welch t-tests.
================================================================================
Reads results/pinn.jsonl and compares each configuration against the
no-physics control. Reports the base-rate INVARIANT metric (AUC) beside the
base-rate DEPENDENT one (F1), because this project has already shown that F1
can move a long way without any change in real skill.

    python analyse_pinn.py --file results/pinn.jsonl
"""
import argparse, json, math
from collections import defaultdict
import numpy as np

ap = argparse.ArgumentParser()
ap.add_argument("--file", default="results/pinn.jsonl")
ap.add_argument("--min-seeds", type=int, default=2)
a = ap.parse_args()

rows = [json.loads(l) for l in open(a.file) if l.strip()]
print(f"{len(rows)} runs in {a.file}\n")

def key(r):
    ph = r.get("physics", "short" if r.get("pde_weight", 1) > 0 else "none")
    return f"{ph:6s} + {r.get('loss','mse'):7s}"

G = defaultdict(list)
for r in rows:
    G[key(r)].append(r)

def ms(v):
    v = [x for x in v if x is not None and not (isinstance(x, float) and math.isnan(x))]
    return (float(np.mean(v)), float(np.std(v)), len(v)) if v else (float("nan"),)*2 + (0,)

def welch(a1, a2):
    a1 = np.asarray([x for x in a1 if x is not None]); a2 = np.asarray([x for x in a2 if x is not None])
    if len(a1) < 2 or len(a2) < 2: return float("nan"), float("nan")
    s1, s2 = a1.var(ddof=1)/len(a1), a2.var(ddof=1)/len(a2)
    se = math.sqrt(s1 + s2)
    if se == 0: return float("nan"), float("nan")
    t = (a1.mean() - a2.mean()) / se
    df = (s1+s2)**2 / (s1**2/(len(a1)-1) + s2**2/(len(a2)-1))
    return t, df

# ---------------------------------------------------------------- main table
print("=" * 84)
print("OVERALL  (mean +/- std over seeds)")
print("=" * 84)
print(f"{'configuration':>18} {'n':>3} {'AUC':>14} {'F1':>14} {'RMSE':>14} {'eta':>9}")
print("-" * 84)
order = sorted(G, key=lambda k: -ms([r['ALL_auc'] for r in G[k]])[0])
for k in order:
    R = G[k]
    if len(R) < a.min_seeds: continue
    au, aus, n = ms([r['ALL_auc'] for r in R])
    f1, f1s, _ = ms([r['ALL_f1'] for r in R])
    rm, rms, _ = ms([r['rmse'] for r in R])
    et, ets, _ = ms([r.get('eta') for r in R])
    print(f"{k:>18} {n:3d} {au:8.2f}±{aus:5.2f} {f1:8.2f}±{f1s:5.2f} "
          f"{rm:8.3f}±{rms:5.3f} {et:9.4f}")

# ---------------------------------------------------------------- vs control
ctrl = next((k for k in G if k.strip().startswith("none")), None)
if ctrl:
    print("\n" + "=" * 84)
    print(f"VERSUS CONTROL  ({ctrl.strip()})   ** = p<0.05")
    print("=" * 84)
    print(f"{'configuration':>18} {'dAUC':>8} {'t':>8} {'sig':>4} "
          f"{'dF1':>8} {'t':>8} {'sig':>4}")
    print("-" * 66)
    for k in order:
        if k == ctrl or len(G[k]) < a.min_seeds: continue
        for name, fld in (("auc", "ALL_auc"), ("f1", "ALL_f1")):
            pass
        t1, d1 = welch([r['ALL_auc'] for r in G[k]], [r['ALL_auc'] for r in G[ctrl]])
        t2, d2 = welch([r['ALL_f1'] for r in G[k]], [r['ALL_f1'] for r in G[ctrl]])
        crit = 2.78
        da = ms([r['ALL_auc'] for r in G[k]])[0] - ms([r['ALL_auc'] for r in G[ctrl]])[0]
        df_ = ms([r['ALL_f1'] for r in G[k]])[0] - ms([r['ALL_f1'] for r in G[ctrl]])[0]
        print(f"{k:>18} {da:+8.2f} {t1:8.2f} {'**' if abs(t1)>crit else '':>4} "
              f"{df_:+8.2f} {t2:8.2f} {'**' if abs(t2)>crit else '':>4}")

# ---------------------------------------------------------------- the point
print("\n" + "=" * 84)
print("DOES THE FAIRNESS METRIC TRACK REAL SKILL?")
print("=" * 84)
print(f"{'configuration':>18} {'pooled AUC':>13} {'Head-Tail F1 gap':>19} {'AUC gap':>11}")
print("-" * 66)
aucs, gaps = [], []
for k in order:
    if len(G[k]) < a.min_seeds: continue
    au = ms([r['ALL_auc'] for r in G[k]])[0]
    gf, gfs, _ = ms([r['gap_f1'] for r in G[k]])
    ga, gas, _ = ms([r['gap_auc'] for r in G[k]])
    aucs.append(au); gaps.append(gf)
    print(f"{k:>18} {au:13.2f} {gf:12.2f}±{gfs:5.2f} {ga:+7.2f}±{gas:4.2f}")
if len(aucs) > 1:
    print(f"\n  real skill (AUC) spans      {max(aucs)-min(aucs):6.2f} points")
    print(f"  the F1 'fairness gap' spans {max(gaps)-min(gaps):6.2f} points")
    print("\n  If the second number is small while the first is large, the metric a")
    print("  fairness audit would report is insensitive to genuine model quality.")

# ---------------------------------------------------------------- tail detail
print("\n" + "=" * 84)
print("TAIL REGIONS  (crime actually occurs in ~28.5% of weeks)")
print("=" * 84)
print(f"{'configuration':>18} {'Tail F1':>14} {'Tail AUC':>14} {'says yes':>14} {'x too often':>12}")
print("-" * 78)
for k in order:
    if len(G[k]) < a.min_seeds: continue
    f1, f1s, _ = ms([r['Tail_f1'] for r in G[k]])
    au, aus, _ = ms([r['Tail_auc'] for r in G[k]])
    ry, rys, _ = ms([r['Tail_rate'] for r in G[k]])
    br = ms([r['Tail_base'] for r in G[k]])[0]
    print(f"{k:>18} {f1:8.2f}±{f1s:5.2f} {au:8.2f}±{aus:5.2f} "
          f"{ry:8.1f}±{rys:4.1f}% {ry/max(br,1e-9):11.1f}x")
print("=" * 84)


In [ ]:
!wc -l build_field.py crime_pinn.py signal_check.py analyse_pinn.py
# crime_pinn.py must be 481 lines and contain the Tier-1 markers:
!grep -c "A0Field\|smooth_residuals\|u_raw_t" crime_pinn.py   # expect 7

## 3. Build the field

Skipped automatically if it already exists in Drive. ~3 minutes the first time.
94,992 burglaries, 2011–2015, 24x24 grid, weekly. 183 training weeks, ~41% empty.

In [ ]:
import os
if not os.path.exists('data/burg_w24.npz'):
    !python build_field.py --year0 2011 --year1 2015 --freq W --grid 24 \
        --category BURGLARY --smooth 1.2 --out data/burg_w24.npz
else:
    print('data/burg_w24.npz already built')

## 4. Sanity check — is there anything to predict?

Compares three trivial predictors. If "map + last week" barely beats "the map",
there is no week-to-week signal and the models can only learn geography.

In [ ]:
!python signal_check.py --data data/burg_w24.npz

## 5. The experiment — 15 runs

Three configurations x 5 seeds:

* **short + poisson** — Short et al. PDE, correct count likelihood
* **short + mse** — same physics, Gaussian loss (the older setting)
* **none + poisson** — capacity-matched control, physics switched off

~45 minutes. Results append to `results/pinn.jsonl` in Drive, so a disconnect
mid-loop only costs the runs not yet finished.

In [ ]:
for s in range(5):
    !python crime_pinn.py --data data/burg_w24.npz --physics short --loss poisson --seed {s} --save results/pinn.jsonl
    !python crime_pinn.py --data data/burg_w24.npz --physics short --loss mse     --seed {s} --save results/pinn.jsonl
    !python crime_pinn.py --data data/burg_w24.npz --physics none  --loss poisson --seed {s} --save results/pinn.jsonl
print('\nDONE —', sum(1 for _ in open('results/pinn.jsonl')), 'runs recorded')

## 6. The final table

In [ ]:
!python analyse_pinn.py --file results/pinn.jsonl

### What to look for

**Does the physics help?** Compare `short + poisson` against `none + poisson` on
AUC. Earlier five-seed runs gave roughly +18 points with t > 10.

**Does it over-predict in poor areas?** The Tail table shows how often each model
says "crime" against the real rate of 28.5%. The control was ~3.2x too often;
physics plus the Poisson likelihood brought that to ~1.8x.

**The headline.** Compare how far pooled AUC spans across the three
configurations against how far the Head−Tail F1 gap spans. Earlier: AUC spanned
18.4 points, the F1 gap spanned 1.8. A model near chance and a model with real
skill report the same fairness number.